# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

The goal of this baseline is to rank pages by their likelihood of needing attention in the following month.

Unlike the original version, this baseline uses historical monthly behavior instead of a single monthly snapshot.

The rule uses only information available up to the current month.

Scoring rule:

- Add 2 points if impression growth from the previous month is below -20%.
- Add 2 points if average position is greater than 20.
- Add 1 point if 3-month average impressions are lower than the previous month's impressions.

Action mapping:

- Score 5 → High Risk (Refresh)
- Score 3–4 → Medium Risk (Review)
- Score 1–2 → Low Risk (Monitor)
- Score 0 → Stable (No Action)

### Reason Codes

- DECLINING_IMPRESSIONS
- LOW_POSITION
- WEAK_3_MONTH_TREND
- STABLE

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline Scoring

Each page-month record is assigned a risk score using only information available up to the current month.

Scoring rules:

- +2 points if impression growth from the previous month is less than -20%.
- +2 points if average position is greater than 20.
- +1 point if the current month's impressions are below the 3-month rolling average.

The total score ranges from 0 to 5.

Action mapping:

| Score | Action |
|-------|--------|
| 5 | Refresh |
| 3–4 | Review |
| 1–2 | Monitor |
| 0 | No Action |

The ranked queue is sorted by score in descending order and exported as:

`work/outputs/baseline_action_score.csv`

This file serves as the Week-4 baseline and will be compared against the machine learning model in Week 5.

In [4]:
import numpy as np
from pathlib import Path

# ==========================================================
# Clean data
# ==========================================================
import pandas as pd

page_month = pd.read_parquet(
    r"C:\Users\ahmad\Documents\GitHub\FlyRankML\work\outputs\page_month.parquet"
)
page_month = page_month.copy()

page_month = page_month.replace([np.inf, -np.inf], np.nan)
page_month = page_month.fillna(0)

# ==========================================================
# Baseline Score
# ==========================================================

page_month["score"] = 0

# Large drop compared to previous month
page_month.loc[
    page_month["impression_growth"] < -0.20,
    "score"
] += 2

# Poor ranking
page_month.loc[
    page_month["avg_position"] > 20,
    "score"
] += 2

# Current impressions below recent trend
page_month.loc[
    page_month["impressions"] < page_month["rolling3_impressions"],
    "score"
] += 1

# ==========================================================
# Reason Codes
# ==========================================================

conditions = [
    page_month["score"] >= 5,
    page_month["score"] >= 3,
    page_month["score"] >= 1
]

choices = [
    "HIGH_DECLINE_RISK",
    "MEDIUM_DECLINE_RISK",
    "LOW_DECLINE_RISK"
]

page_month["reason_code"] = np.select(
    conditions,
    choices,
    default="STABLE"
)

# ==========================================================
# Recommended Action
# ==========================================================

conditions = [
    page_month["score"] >= 5,
    page_month["score"] >= 3,
    page_month["score"] >= 1
]

choices = [
    "Refresh",
    "Review",
    "Monitor"
]

page_month["action"] = np.select(
    conditions,
    choices,
    default="No Action"
)

# ==========================================================
# Rank Queue
# ==========================================================

baseline_queue = (
    page_month
    .sort_values(
        ["score", "impression_growth", "impressions"],
        ascending=[False, True, False]
    )
    .reset_index(drop=True)
)

# ==========================================================
# Save CSV
# ==========================================================

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("Saved!")
print(baseline_queue.shape)

print(pd.read_csv("work/outputs/baseline_action_score.csv").head())
# ==========================================================
# Summary
# ==========================================================

print("\nScore Distribution")
print(baseline_queue["score"].value_counts().sort_index())

print("\nReason Codes")
print(baseline_queue["reason_code"].value_counts())

print("\nActions")
print(baseline_queue["action"].value_counts())



Saved!
(1329512, 21)
            content_hash_id    month  impressions  clicks  avg_position  \
0  content_4386a049011ab551  2025-08          1.0     0.0          96.0   
1  content_96fc4785fcacf0ca  2025-08          2.0     0.0          65.5   
2  content_5f773cc42ddd6d11  2025-04          1.0     0.0          95.0   
3  content_a159e2aafb79cab0  2025-08          1.0     0.0          76.0   
4  content_c49b04c2557dc5b9  2025-07          1.0     0.0          81.0   

   organic_sessions  next_month_impressions  impression_change  \
0               0.0                     1.0                0.0   
1               0.0                     2.0                0.0   
2               0.0                  3332.0             3331.0   
3               0.0                     1.0                0.0   
4               0.0                   987.0              986.0   

   needs_attention  prev_impressions  ...  prev_position  impression_growth  \
0                0            3776.0  ...      79.12

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [2]:
# ==========================================================
# Top 20 Queue
# ==========================================================

top20 = baseline_queue[
    [
        "content_hash_id",
        "month",
        "impressions",
        "clicks",
        "avg_position",
        "impression_growth",
        "rolling3_impressions",
        "score",
        "reason_code",
        "action",
    ]
].head(20)

display(top20)
top20

,content_hash_id,month,impressions,clicks,avg_position,impression_growth,rolling3_impressions,score,reason_code,action
0,content_4386a049011ab551,2025-08,1.0,0.0,96.000000,-0.999735,1267.666667,5,HIGH_DECLINE_RISK,Refresh
1,content_96fc4785fcacf0ca,2025-08,2.0,0.0,65.500000,-0.999582,3430.333333,5,HIGH_DECLINE_RISK,Refresh
2,content_5f773cc42ddd6d11,2025-04,1.0,0.0,95.000000,-0.999381,808.000000,5,HIGH_DECLINE_RISK,Refresh
3,content_a159e2aafb79cab0,2025-08,1.0,0.0,76.000000,-0.999302,716.500000,5,HIGH_DECLINE_RISK,Refresh
4,content_c49b04c2557dc5b9,2025-07,1.0,0.0,81.000000,-0.999246,1353.333333,5,HIGH_DECLINE_RISK,Refresh
5,content_e6967b21fd4db5dc,2026-02,1.0,0.0,49.000000,-0.999113,913.333333,5,HIGH_DECLINE_RISK,Refresh
6,content_88fc8c8ca7a01d66,2025-04,1.0,0.0,49.000000,-0.998929,467.500000,5,HIGH_DECLINE_RISK,Refresh
7,content_130b440ff31666ab,2026-01,1.0,0.0,99.000000,-0.998926,574.000000,5,HIGH_DECLINE_RISK,Refresh
8,content_5b4032e0d1bc5ead,2025-08,1.0,0.0,43.000000,-0.998904,456.500000,5,HIGH_DECLINE_RISK,Refresh
9,content_7b09ab992bbed064,2026-01,1.0,0.0,27.000000,-0.998875,381.000000,5,HIGH_DECLINE_RISK,Refresh


,content_hash_id,month,impressions,clicks,avg_position,impression_growth,rolling3_impressions,score,reason_code,action
0,content_4386a049011ab551,2025-08,1.0,0.0,96.000000,-0.999735,1267.666667,5,HIGH_DECLINE_RISK,Refresh
1,content_96fc4785fcacf0ca,2025-08,2.0,0.0,65.500000,-0.999582,3430.333333,5,HIGH_DECLINE_RISK,Refresh
2,content_5f773cc42ddd6d11,2025-04,1.0,0.0,95.000000,-0.999381,808.000000,5,HIGH_DECLINE_RISK,Refresh
3,content_a159e2aafb79cab0,2025-08,1.0,0.0,76.000000,-0.999302,716.500000,5,HIGH_DECLINE_RISK,Refresh
4,content_c49b04c2557dc5b9,2025-07,1.0,0.0,81.000000,-0.999246,1353.333333,5,HIGH_DECLINE_RISK,Refresh
5,content_e6967b21fd4db5dc,2026-02,1.0,0.0,49.000000,-0.999113,913.333333,5,HIGH_DECLINE_RISK,Refresh
6,content_88fc8c8ca7a01d66,2025-04,1.0,0.0,49.000000,-0.998929,467.500000,5,HIGH_DECLINE_RISK,Refresh
7,content_130b440ff31666ab,2026-01,1.0,0.0,99.000000,-0.998926,574.000000,5,HIGH_DECLINE_RISK,Refresh
8,content_5b4032e0d1bc5ead,2025-08,1.0,0.0,43.000000,-0.998904,456.500000,5,HIGH_DECLINE_RISK,Refresh
9,content_7b09ab992bbed064,2026-01,1.0,0.0,27.000000,-0.998875,381.000000,5,HIGH_DECLINE_RISK,Refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

The baseline is intentionally simple and rule-based, so some pages may receive incorrect recommendations.

### Weak picks

Some pages receive a high score because they experienced a temporary drop in impressions or already have poor average rankings. These pages are not necessarily declining permanently. Seasonal traffic, news events, or temporary search fluctuations can cause the baseline to overestimate decline risk.

Similarly, pages with stable impressions but improving rankings may still receive unnecessary review recommendations because the baseline does not understand longer-term trends.

The baseline also cannot distinguish between naturally low-performing pages and pages that genuinely require optimization.

### Leakage check

Only information available at or before the current month was used to calculate the score.

The baseline uses:

- Current month's impressions
- Previous month's impressions
- Previous month's clicks
- Current average position
- Three-month rolling averages

No future month's impressions, clicks, rankings, or sessions were used when assigning scores.

No product flags, client-specific information, or manually assigned future labels were included in the scoring process.

Therefore, the baseline is free from future data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.